```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2,A3,A4a done;
    class A4b current;
    class A5a,A5b,A6a,A6b,A7,A8a,A8b,A9,A10,A11,A12 normal;
```

# Notebook 04b — Lexical diversity, association, and exploratory representations II (embedding)

### _Embeddings_

TF–IDF represents texts through **weighted word/phrase overlap**. Embeddings offer a complementary view: they map each text (here, short paragraph “chunks”) to a **dense vector** such that passages with similar meaning tend to be closer in the vector space, even when they do not share many exact words. This makes embeddings useful for *semantic* exploration tasks like nearest-neighbor search (“find passages like this query”) and for comparing **periods** by averaging vectors within each time bin.

In this section we:
1) split each book into paragraph-sized chunks (so the model sees manageable units),
2) compute and **cache** chunk embeddings (so we do this once),
3) run a few semantic search queries aligned with the course theme,
4) visualize similarity across time bins using **centroid embeddings**.

**Method note:** embedding dimensions are not interpretable one-by-one. We interpret embeddings through (a) similarity patterns, and (b) representative passages retrieved from neighborhoods—always with caution about corpus composition and date proxies.



In [ ]:
!pip install sentence-transformers

In [ ]:
# -----------------------------
# Import
# -----------------------------

import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity

# Optional but often reduces tokenizer/threading issues in notebooks
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    from sentence_transformers import SentenceTransformer
    from sklearn.neighbors import NearestNeighbors
    HAS_ST = True
except Exception as e:
    HAS_ST = False
    print("sentence-transformers not available; skipping embedding section.")
    print("Install with: pip install sentence-transformers scikit-learn")
    print("Import error:", e)

In [ ]:
# -----------------------------
# Paths
# -----------------------------

PROJECT_ROOT = Path(".")  # run from the Notebooks/ folder

# Output and cache folders
OUTPUT_DIR = PROJECT_ROOT / "analysis"
CACHE = PROJECT_ROOT / "cache" 

TEXTS_DIR = PROJECT_ROOT / "data" / "processed" / "cleaned"
TABLES_DIR = Path(OUTPUT_DIR) / "tables"
FIGURES_DIR = Path(OUTPUT_DIR) / "figures"
CACHE_DIR = Path(CACHE)

In [ ]:
def chunk_paragraphs(text: str) -> list[str]:
    """Split into paragraph chunks, drop very short ones, trim very long ones."""
    paras = [p.strip() for p in re.split(r"\n\s*\n+", str(text)) if p.strip()]
    chunks: list[str] = []
    for p in paras:
        if len(p) < MIN_CHUNK_CHARS:
            continue
        if len(p) > MAX_CHUNK_CHARS:
            p = p[:MAX_CHUNK_CHARS]
        chunks.append(p)
    return chunks

In [ ]:
# -----------------------------
# Load documents' metadata
# -----------------------------
doc_index = TABLES_DIR / "nb03-doc_index.csv"
print("\nLoading document table from:", doc_index)
df = pd.read_csv(doc_index)
df["publication_year"] = pd.to_numeric(df["publication_year"], errors="coerce").astype("Int64")
print(f"\n{len(df)} documents' metadata loaded.")

In [ ]:
# -----------------------------
# Store texts in df
# -----------------------------
def read_clean_text(texts_dir: Path, pg_id: int) -> str:
    filename = f"pg{pg_id}.txt"
    return (texts_dir / filename).read_text(encoding="utf-8", errors="replace")
    
# Create a new column and store corresponding texts
df["text"] = df["pg_id"].apply(lambda pid: read_clean_text(TEXTS_DIR, pid))
df["publication_year"] = df["publication_year"].astype("Int64")

# Expected columns ["pg_id", "title", "publication_year", "time_bin", "text"]
df.columns.to_list()

In [ ]:
# -----------------------------
# Parameters
# -----------------------------
CHUNK_MODE = "paragraph"   # "paragraph" or "sentences"
MIN_CHUNK_CHARS = 300
MAX_CHUNK_CHARS = 1500     # soft cap; long paragraphs are trimmed
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

In [ ]:
# ============================================================
# TEST
# Builds a very small chunk sample and checks
# that the embedding model loads and can encode a few chunks.
# ============================================================

if not HAS_ST:
    raise RuntimeError("sentence-transformers is not available in this environment.")

required_cols = {"pg_id", "title", "publication_year", "time_bin", "text"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"df is missing required columns: {sorted(missing_cols)}")

print("Preparing a small sample for testing...")

sample_chunk_rows = []
max_books_for_test = 8
max_chunks_for_test = 64

for r in df[["pg_id", "title", "publication_year", "time_bin", "text"]].head(max_books_for_test).itertuples(index=False):
    pg_id_val = int(r.pg_id) if pd.notna(r.pg_id) else pd.NA
    for k, ch in enumerate(chunk_paragraphs(r.text)):
        sample_chunk_rows.append({
            "pg_id": pg_id_val,
            "title": r.title,
            "publication_year": r.publication_year,
            "time_bin": r.time_bin,
            "chunk_id": f"{pg_id_val}_{k}",
            "text": ch,
            "chunk_n_chars": len(ch),
        })
        if len(sample_chunk_rows) >= max_chunks_for_test:
            break
    if len(sample_chunk_rows) >= max_chunks_for_test:
        break

sample_chunks_df = pd.DataFrame(sample_chunk_rows)

if sample_chunks_df.empty:
    raise ValueError(
        "No chunks were created in the smoke test. Try lowering MIN_CHUNK_CHARS or check the input texts."
    )

print(f"Smoke-test chunks: {len(sample_chunks_df):,}")
print("Loading model:", EMBEDDING_MODEL_NAME)

model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = sample_chunks_df["text"].astype(str).tolist()
t0 = time.time()
E_test = model.encode(
    texts,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
elapsed = time.time() - t0

print(f"Smoke test complete: {E_test.shape[0]:,} embeddings with dimension {E_test.shape[1]}")
print(f"Elapsed time: {elapsed:.1f} seconds")
print(f"Average per chunk: {elapsed / len(texts):.3f} seconds")

# Optional: quick semantic search check on the tiny sample
nn_test = NearestNeighbors(n_neighbors=min(5, len(E_test)), metric="cosine").fit(E_test)
query = "freedom and justice"
q_vec = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
dists, idx = nn_test.kneighbors(q_vec, n_neighbors=min(5, len(E_test)))

smoke_out = sample_chunks_df.iloc[idx[0]].copy()
smoke_out["cosine_sim"] = 1 - dists[0]
print("\nTop smoke-test matches for query:", query)
display(smoke_out[["cosine_sim", "title", "publication_year", "time_bin", "chunk_id", "chunk_n_chars", "text"]])

In [ ]:
# ============================================================
# FULL EMBEDDING PIPELINE
# Run this only if the smoke test above works and
# the runtime seems acceptable
# ============================================================

if not HAS_ST:
    raise RuntimeError("sentence-transformers is not available in this environment.")

required_cols = {"pg_id", "title", "publication_year", "time_bin", "text"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"df is missing required columns: {sorted(missing_cols)}")

TABLES_DIR = Path(OUTPUT_DIR) / "tables"
FIGURES_DIR = Path(OUTPUT_DIR) / "figures"
CACHE_DIR = Path(CACHE)

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

chunks_path = TABLES_DIR / "nb04-chunks.parquet"
chunks_index_path = TABLES_DIR / "nb04-chunks_index.csv"
emb_path = CACHE_DIR / "nb04-chunk_embeddings.npy"

# -----------------------------
# SECTION A — Build or load chunks
# -----------------------------
if chunks_path.exists():
    print("Loading cached chunks table...")
    chunks_df = pd.read_parquet(chunks_path)
else:
    print("Building paragraph-level chunks...")
    chunk_rows = []

    for r in tqdm(
        df[["pg_id", "title", "publication_year", "time_bin", "text"]].itertuples(index=False),
        total=len(df),
        desc="Books"
    ):
        pg_id_val = int(r.pg_id) if pd.notna(r.pg_id) else pd.NA
        for k, ch in enumerate(chunk_paragraphs(r.text)):
            chunk_rows.append({
                "pg_id": pg_id_val,
                "title": r.title,
                "publication_year": r.publication_year,
                "time_bin": r.time_bin,
                "chunk_id": f"{pg_id_val}_{k}",
                "text": ch,
                "chunk_n_chars": len(ch),
            })

    chunks_df = pd.DataFrame(chunk_rows)
    chunks_df["publication_year"] = chunks_df["publication_year"].astype("Int64")
    if chunks_df.empty:
        raise ValueError(
            "No chunks were created. Try lowering MIN_CHUNK_CHARS or inspect the source texts."
        )

    print("Chunks built:", f"{len(chunks_df):,}")
    chunks_df.to_parquet(chunks_path, index=False)
    print("Saved chunk table:", chunks_path)

print("Final chunk count:", f"{len(chunks_df):,}")

# -----------------------------
# SECTION B — Load embedding model once
# -----------------------------
print("Loading embedding model:", EMBEDDING_MODEL_NAME)
model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# -----------------------------
# SECTION C — Build or load embeddings
# -----------------------------
if emb_path.exists():
    print("Loading cached embeddings...")
    E = np.load(emb_path)
else:
    print("Computing embeddings...")
    texts = chunks_df["text"].astype(str).tolist()

    # Small timing probe so you can estimate whether to keep or drop this section
    probe_n = min(64, len(texts))
    print(f"Running timing probe on first {probe_n:,} chunks...")
    t0_probe = time.time()
    _ = model.encode(
        texts[:probe_n],
        batch_size=8,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    probe_elapsed = time.time() - t0_probe
    sec_per_chunk = probe_elapsed / probe_n
    est_total_minutes = (sec_per_chunk * len(texts)) / 60
    print(f"Probe time: {probe_elapsed:.1f}s total | {sec_per_chunk:.3f}s per chunk")
    print(f"Rough estimated full runtime: {est_total_minutes:.1f} minutes")

    # Main embedding pass
    t0 = time.time()
    E = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    total_elapsed = time.time() - t0

    np.save(emb_path, E)
    print("Saved embeddings:", emb_path)
    print(f"Embedding time: {total_elapsed / 60:.1f} minutes")

# -----------------------------
# SECTION D — Sanity checks + lightweight index
# -----------------------------
if len(chunks_df) != len(E):
    raise ValueError(f"Mismatch: chunks_df has {len(chunks_df)} rows but embeddings have {len(E)} vectors")

chunks_df.drop(columns=["text"], errors="ignore").to_csv(chunks_index_path, index=False)
print("Saved chunk index:", chunks_index_path)
print("Embedding matrix shape:", E.shape)

# -----------------------------
# SECTION E — Nearest neighbors for semantic search
# -----------------------------
nn = NearestNeighbors(n_neighbors=10, metric="cosine")
nn.fit(E)


def query_neighbors(query: str, k: int = 8) -> pd.DataFrame:
    """Return the top-k nearest chunk neighbors to a query string."""
    q = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    dists, idx = nn.kneighbors(q, n_neighbors=k)
    out = chunks_df.iloc[idx[0]].copy()
    out["cosine_sim"] = 1 - dists[0]
    return out[[
        "cosine_sim", "pg_id", "title", "publication_year", "time_bin",
        "chunk_id", "chunk_n_chars", "text"
    ]]

# -----------------------------
# SECTION F — Demo queries for class discussion
# -----------------------------
for q in [
    "reason and experience",
    "freedom and justice",
    "nature and virtue",
]:
    print("\nQuery:", q)
    display(query_neighbors(q, k=5))

# Optional cleanup for display only
pd.set_option("display.max_colwidth", 180)


### Visualization: similarity heatmap across time bins (embedding centroids)
We take as basis that **each chunk is a vector**. We aggregate vectors within each time_bin by taking their mean (centroid). We then compare bins via cosine similarity of centroids.

This gives a compact picture of which periods look semantically similar.


In [ ]:
# Chronological sort key for a time_bin label.
# Works for pandas Interval objects (sorts by left edge) and for
# string labels like "1700-1749" or "1700–1749" (en dash).
def bin_sort_key(b):
    if hasattr(b, "left"):
        return float(b.left)
    s = str(b).replace("–", "-").replace("−", "-")
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group(0)) if m else float("inf")


In [ ]:
# -----------------------------
# Build time-bin centroids — single source of truth for C, S, bin_groups, bin_labels
# -----------------------------
# Reload from cache so this cell also works standalone after a kernel restart,
# without needing to re-run the full embedding pipeline above.
chunks_df = pd.read_parquet(chunks_path)
E = np.load(emb_path)
assert len(chunks_df) == len(E), (len(chunks_df), len(E))

# Keep only chunks that have a time_bin, and filter E with the SAME boolean mask
# so row positions in chunks_binned and E_binned stay aligned with each other.
time_bin_mask = chunks_df["time_bin"].notna().to_numpy()
chunks_binned = chunks_df.loc[time_bin_mask].reset_index(drop=True)
E_binned = E[time_bin_mask]

# Row positions (within chunks_binned / E_binned) for each time bin
bin_groups = chunks_binned.groupby("time_bin").indices
bin_labels = sorted(bin_groups.keys(), key=bin_sort_key)

bin_centroids = []
bin_counts = []
for b in bin_labels:
    idx = np.array(list(bin_groups[b]), dtype=int)
    c = E_binned[idx].mean(axis=0)
    c = c / (np.linalg.norm(c) + 1e-12)  # unit-normalize -> cosine sim reduces to a dot product later
    bin_centroids.append(c)
    bin_counts.append(len(idx))

C = np.vstack(bin_centroids)   # n_bins x dim — THE centroid matrix, reused by every cell below
S = cosine_similarity(C)       # n_bins x n_bins
sim_df = pd.DataFrame(S, index=bin_labels, columns=bin_labels)
bin_counts_df = pd.DataFrame({"time_bin": bin_labels, "n_chunks": bin_counts})

# Diagnostics: off-diagonal similarity range (helps pick a sensible color scale below)
S_off = S.copy()
np.fill_diagonal(S_off, np.nan)
print("Bins:", len(bin_labels))
print(bin_counts_df.to_string(index=False))
print("\nCosine similarity off-diagonal:",
      f"min={np.nanmin(S_off):.4f}, median={np.nanmedian(S_off):.4f}, max={np.nanmax(S_off):.4f}")

# --- Plot (tighter color range so small differences are visible) ---
vmin = float(np.nanpercentile(S_off, 5)) if np.isfinite(np.nanmin(S_off)) else 0.0
vmax = 1.0

plt.figure(figsize=(10, 8))
sns.heatmap(sim_df, cmap="crest", vmin=vmin, vmax=vmax)
plt.title(f"Embedding similarity across time bins (centroid cosine)\n(vmin={vmin:.3f}, vmax={vmax:.3f})")
plt.xlabel("Time bin")
plt.ylabel("Time bin")
plt.tight_layout()

fig_path = FIGURES_DIR / "nb04-embedding_timebin_centroid_cosine_heatmap.png"
plt.savefig(fig_path, dpi=200)
plt.show()
print("Saved figure:", fig_path)


### Most similar time-bin pairs
Using the `S` / `bin_labels` / `bin_counts` computed above, list the pairs of time bins whose centroids are most similar.


In [ ]:
# Top most-similar time-bin pairs (reuses S / bin_labels / bin_counts from the cell above — no recomputation)
pairs = []
for i in range(len(bin_labels)):
    for j in range(i + 1, len(bin_labels)):
        pairs.append((
            str(bin_labels[i]), str(bin_labels[j]), float(S[i, j]),
            int(bin_counts[i]), int(bin_counts[j]),
        ))

pairs_df = (
    pd.DataFrame(pairs, columns=["bin_a", "bin_b", "cosine_sim", "n_chunks_a", "n_chunks_b"])
    .sort_values("cosine_sim", ascending=False)
    .head(15)
)
display(pairs_df)


In [ ]:
from sklearn.decomposition import PCA

Z = PCA(n_components=2, random_state=0).fit_transform(C)  # C is n_bins x dim

plt.figure(figsize=(8, 6))
plt.scatter(Z[:, 0], Z[:, 1], s=80)

# label points
for (x, y, lab) in zip(Z[:, 0], Z[:, 1], bin_labels):
    plt.text(x, y, str(lab), fontsize=9)

# connect in chronological order
plt.plot(Z[:, 0], Z[:, 1], linewidth=1)

plt.title("Time-bin embedding centroids (PCA 2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()


In [ ]:
# cosine distance between consecutive bin centroids
# since C rows are L2-normalized, cosine similarity = dot product
cos_sim_next = np.sum(C[1:] * C[:-1], axis=1)
cos_dist_next = 1 - cos_sim_next

shift = pd.DataFrame({
    "time_bin": bin_labels[1:],
    "cosine_distance_to_prev": cos_dist_next
})

plt.figure(figsize=(10, 4))
plt.plot(shift["time_bin"], shift["cosine_distance_to_prev"], marker="o")
plt.xticks(rotation=45, ha="right")
plt.title("Semantic shift by time bin (embedding centroid distance to previous bin)")
plt.ylabel("Cosine distance (1 - cosine similarity)")
plt.xlabel("Time bin")
plt.tight_layout()
plt.show()

display(shift.sort_values("cosine_distance_to_prev", ascending=False).head(10))


# Typical text chunks per time bin

In the next cell, we display the top 5 text chunks that are most representative of a particular time-bin cluster, i.e., the chunks whose embeddings are closest (in cosine similarity) to that cluster's centroid.

We use the variables built above (in the "Build time-bin centroids" cell) — each is now defined exactly once:

`C` — the n_bins × dim matrix of centroid vectors, one row per time bin, unit-normalized.
A centroid is the average of all chunk embeddings that fall into that bin — the "semantic center of gravity" for that period.

`bin_labels` — the ordered list of time-bin names (e.g., `["1600–1650", "1651–1700", …]`), sorted chronologically. The position of a label in this list matches the row of `C`.

`bin_groups` — a dict mapping each time-bin label to the row positions (within `chunks_binned` / `E_binned`) of the chunks that belong to it.

`chunks_binned` / `E_binned` — the chunk table and embedding matrix, filtered to rows with a non-null `time_bin`, and kept aligned with each other by construction.

`b = bin_labels[0]` — we pick the first time bin as an example; change this to inspect any other bin.


In [ ]:
b = bin_labels[0]                                    # Select a time bin
idx = np.array(list(bin_groups[b]), dtype=int)       # Row positions (in chunks_binned/E_binned) of chunks in that bin
cent = C[bin_labels.index(b)]                        # Centroid for that bin (mean embedding of those chunks)

# Cosine similarity of each chunk in the bin to the centroid.
# Cosine similarity ranges from -1 to 1; higher means closer in "semantic direction".
sims = cosine_similarity(E_binned[idx], cent.reshape(1, -1)).ravel()

# The 5 chunks whose embeddings lie closest to the center of the cluster
top = idx[np.argsort(-sims)[:5]]

display(chunks_binned.iloc[top][["time_bin", "title", "pg_id", "chunk_id", "text"]])

# Interpretations

Qualitative interpretation of abstract vectors (embeddings) can feel like black boxes. Showing the chunks closest to a centroid turns a numeric representation into something more interpretable.

We have shown here chunks that are typical (central), not necessarily the ones that most distinguish one period from another.

By changing `b` to other labels, we can quickly pull out representative passages for each era to see a snapshot of how language and ideas shift across time.

---

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A4b highlight;
```